# Infernal

Passo 1: Classificação da Sequência (A Busca na Rfam)

O primeiro objetivo é descobrir a qual família de RNA a sequência do usuário pertence.

- Ação: O sistema submete a sequência de RNA do usuário ao "motor de busca" da Rfam (a ferramenta por trás disso se chama Infernal).

- Processo: O Infernal compara a sequência do usuário com sua base de dados de Modelos de Covariância (CMs), onde cada modelo representa uma família Rfam.

- Saída: O resultado é uma lista das famílias mais prováveis (rfam_acc) com suas respectivas pontuações (bit_score, E-value).

- Decisão: O sistema seleciona o melhor resultado (geralmente o que tem o menor E-value e maior bit_score) como a família mais provável para a sequência do usuário.

In [46]:
import subprocess
import os
import tempfile

Busca uma sequência de RNA na base de dados Rfam usando o Infernal (cmscan).
Args:
    sequence (str): A sequência de RNA a ser buscada (ex: "GCGGAUUUAGCUCAG").
    rfam_db_path (str): O caminho para o arquivo da base de dados Rfam.cm preparada.

Returns:
    list: Uma lista de dicionários, onde cada dicionário contém os
          detalhes de uma família de RNA correspondente. Retorna uma
          lista vazia se nenhuma correspondência significativa for encontrada.


In [ ]:
def Infernal(sequence: str, rfam_db_path: str) -> list:

    if not os.path.exists(rfam_db_path):
        raise FileNotFoundError(f"Erro: A base de dados Rfam não foi encontrada em '{rfam_db_path}'.")

    fasta_input = f">user_sequence\n{sequence}\n"

    hits = []

    # Usamos um arquivo temporário para garantir que funcione em todos os sistemas (Linux, Windows, macOS)
    with tempfile.NamedTemporaryFile(mode='w+', delete=True, suffix=".txt") as tblout_file:

        # --- Comando Corrigido ---
        # --tblout [arquivo]: Escreve a saída tabular para um arquivo.
        # --noali: Não calcula alinhamentos, tornando a busca muito mais rápida.
        command = [
            'cmscan',
            '--noali', # Otimização de velocidade
            '--tblout', tblout_file.name,
            rfam_db_path,
            '-'
        ]

        try:
            subprocess.run(
                command,
                input=fasta_input,
                capture_output=True, # Ainda capturamos para ver os erros
                text=True,
                check=True
            )
        except FileNotFoundError:
            raise RuntimeError("Erro: O comando 'cmscan' não foi encontrado.")
        except subprocess.CalledProcessError as e:
            raise RuntimeError(f"O Infernal retornou um erro:\n{e.stderr}")

        # --- Análise (Parse) Corrigida do Resultado Tabular ---
        # Rebobinamos o arquivo para o início para poder lê-lo
        tblout_file.seek(0)

        for line in tblout_file:
            if line.strip().startswith('#'):
                continue

            parts = line.split()
            
            # A saída tabular tem um formato fixo e diferente da saída padrão.
            # Verificamos se a linha tem colunas suficientes.
            if len(parts) < 18:
                continue
        
            # As colunas corretas para a saída --tblout são:
            # 0: target name, 1: accession, 4: E-value, 5: score, 6: bias
            hit_data = {
                'target_name': parts[0],
                'rfam_acc': parts[1],
                'e_value': float(parts[15]),
                'bit_score': float(parts[14]),
                'bias': float(parts[13])      
            }
            hits.append(hit_data)
    
    hits.sort(key=lambda x: x['e_value'])
    return hits

In [ ]:
user_sequence="GCCUGGCGGCCGUAGCGCGGUGGUCCCACCUGACCCCAUGCCGAACUCAGAAGUGAAACGCCGUAGCGCCGAUGGUAGUGUGGGGUCUCCCCAUGCGAGAGUAGGGAACUGCCAGGCAU"
rfam_db_path = "../data/Cm/Rfam.cm" 

try:
  results = Infernal(user_sequence, rfam_db_path)
  
  if results:
    print("\nFamílias correspondentes encontradas (ordenadas por E-value):")
    for hit in results:
        print(
            f"Família: {hit['target_name']} ({hit['rfam_acc']}), "
            f"E-value: {hit['e_value']:.2e}, "
            f"Score: {hit['bit_score']}"
        )
  else:
      print("\nNenhuma família correspondente significativa foi encontrada.")
      
except (FileNotFoundError, RuntimeError) as e:
    print(f"\nERRO: {e}")

Executando a busca no Infernal (usando saída tabular)...

✅ Famílias correspondentes encontradas (ordenadas por E-value):
Família: 5S_rRNA (RF00001), E-value: 1.80e-21, Score: 92.8
Família: DUF805b (RF02914), E-value: 3.30e+00, Score: 9.3
